# Notebook 03 — Independent Validation

This is the notebook that **owns the validation verdict**. Everything downstream (NB04, NB05) inherits from what happens here. Two design choices matter for interpretation:

- NB03 does not retrain. It loads NB02's pickles and **recomputes metrics independently** from the same predictions. The AUC reconciliation check (Phase 4) confirms NB03's 0.6307 matches NB02's to within 1e-6. That is the evidence of independent implementation.

- NB03 asserts NB02 did not write a verdict (Phase 3) before proceeding. If NB02's JSON contained a 'risk_score', NB03 would refuse to run. This is the chain-of-custody discipline.

In [1]:
# Notebook 03 — Independent Validation
# Version 2.1  [EDIT E-M03-VER] bumped from 2.0
# Changes vs 2.0:
#   E-M03-VER    version bump
#   E-M03-01     reads NB01 + NB02 contracts; no silent defaults
#   E-M03-02     no "INVALID" claim; verdict stated as coded rating (A1)
#   E-M03-03     rejection reason correctly attributed (A2)
#   E-M03-04     fairness test on region grouping, FD-05/FD-03 enforced (A3)
#   E-M03-05     PSI reports not_assessed on index cohorts (B3)
#   E-M03-06     KS result elevated to finding when <0.3 (B4)
#   E-M03-07     HL p-value suspicious check carried forward (B5)
#   E-M03-08     group sufficiency N>=30 enforced before DI (C2)
#   E-M03-09     every console token has a findings entry (C7)
#   E-M03-10     performance_summary_v2.json is NB03's output (not NB02's)
#
# NOTE: This is the notebook that OWNS the validation verdict. Everything downstream (NB04, NB05) inherits from what happens here. The file
# performance_summary_v2.json is the contract that makes that inheritance possible.
# 
# What breaks if you change this notebook without updating downstream:
# - NB04 asserts `authored_by == "NB03"` at Phase 2. If you change that field name or value, NB04 refuses to run.
# - NB04 asserts sum(risk_score_components.values()) == risk_score. If the components don't sum, NB04 refuses to run.
# - NB05 reads fairness_validation.csv for the Lending Club baseline DI. If you rename columns or change what's saved, NB05 breaks.
# - The findings register (validation_findings_register.csv) has finding IDs F-03.01 through F-03.11. NB04 reads them. Renumbering breaks the
#   Remediation Plan mapping.
# 
# What this notebook does NOT do:
# - Retrain the model. NB02 does that. NB03 loads pickles from NB02 and re-evaluates them independently.
# - Produce a fairness verdict at per-state one-hot level. That would be the wrong test. FD-05 requires low-cardinality grouping.
# - Inherit an existing verdict. It reads NB02's metrics, computes its own verdict from the frozen DCR component rules, and writes both to the same JSON.

### PHASE 1: ENVIRONMENT SETUP

In [2]:
print("\n" + "=" * 80)
print("PHASE 1: ENVIRONMENT SETUP & REPRODUCIBILITY")
print("=" * 80)

# >> NOTE: Same version-bump discipline as NB01 and NB02. NB03's version
# >> appears in the JSON it produces (via run_id and notebook_version fields)
# >> and in the findings register (via run_id / notebook_version columns).
# >> If you change the verdict logic (component rules, rating thresholds),
# >> bump the version and re-run all downstream notebooks.
NOTEBOOK_VERSION = "2.1"                                    # [EDIT E-M03-VER]

# >> NOTE: The seed is used here only for SHAP sampling (the first 100
# >> validation rows are deterministic, so the seed is mostly cosmetic).
# >> NB03 does not retrain, so the seed doesn't affect the models. It does
# >> affect any stochastic evaluation — but nothing here is stochastic
# >> except potentially the SHAP sample. The seed is documented for
# >> consistency with the rest of the pipeline.
RANDOM_STATE = 42

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import os
import warnings
import sys
import logging
import hashlib
from pathlib import Path
from datetime import datetime
from scipy import stats
from scipy.stats import chi2

warnings.filterwarnings("ignore")

from sklearn.metrics import (
    roc_auc_score, roc_curve, accuracy_score, precision_score,
    recall_score, f1_score, brier_score_loss, average_precision_score,
    log_loss,
)
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
import shap

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 50)

os.makedirs("outputs/reports", exist_ok=True)
os.makedirs("outputs/figures", exist_ok=True)


def setup_audit_logger(run_id=None):
    """Same pattern as NB01/NB02. Distinct log file per run.

    >> NOTE: The log filename is validation_audit_trail_{run_id}.log, not
    >> model_dev_audit_trail or eda_audit_trail. This is intentional — a
    >> reviewer can find the right log by notebook without scanning file
    >> contents. Match the pattern: {stage}_audit_trail_{run_id}.log.
    """
    if run_id is None:
        run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    os.makedirs("logs", exist_ok=True)
    logger = logging.getLogger(f"lending_club_nb03_{run_id}")
    logger.setLevel(logging.INFO)
    if logger.hasHandlers():
        logger.handlers.clear()
    fh = logging.FileHandler(f"logs/validation_audit_trail_{run_id}.log", encoding="utf-8")
    fh.setFormatter(logging.Formatter(
        '%(asctime)s | %(levelname)s | %(message)s', datefmt='%Y-%m-%d %H:%M:%S'))
    logger.addHandler(fh)
    ch = logging.StreamHandler(sys.stdout)
    ch.setFormatter(fh.formatter)
    logger.addHandler(ch)
    return logger, run_id


def compute_file_hash(path):
    sha256 = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            sha256.update(chunk)
    return sha256.hexdigest()[:16]


def _write_json(obj, path):
    """Write JSON with hash logged.

    >> NOTE: performance_summary_v2.json is the single most important
    >> artifact this notebook writes. It is read by NB04 (verdict inheritance)
    >> and NB05 (fairness baseline). The hash is logged so a reviewer can
    >> verify the file hasn't been edited between runs.
    >>
    >> Any change to this file's schema is a MAJOR version bump. Adding a
    >> field is safe (downstream ignores unknown keys). Removing or renaming
    >> a field breaks NB04/NB05.
    """
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)
    audit_logger.info(f"Saved: {path} hash={compute_file_hash(path)}")


audit_logger, RUN_ID = setup_audit_logger()
audit_logger.info("=" * 60)
audit_logger.info(f"NOTEBOOK 03 v{NOTEBOOK_VERSION}: INDEPENDENT VALIDATION")
audit_logger.info(f"RUN ID: {RUN_ID}")
audit_logger.info("=" * 60)

print(f"[OK] Random seed: {RANDOM_STATE}")
print(f"[OK] Run ID: {RUN_ID}  |  Notebook version: {NOTEBOOK_VERSION}")
np.random.seed(RANDOM_STATE)


PHASE 1: ENVIRONMENT SETUP & REPRODUCIBILITY
2026-10-22 12:30:50 | INFO | ============================================================
2026-10-22 12:30:50 | INFO | NOTEBOOK 03 v2.1: INDEPENDENT VALIDATION
2026-10-22 12:30:50 | INFO | RUN ID: 20261022_123050
2026-10-22 12:30:50 | INFO | ============================================================
[OK] Random seed: 42
[OK] Run ID: 20261022_123050  |  Notebook version: 2.1


### PHASE 2: DATASET CAVEAT (REFERENCED)

In [3]:
# >> NOTE: The caveat is referenced, not repeated. Full caveat lives in
# >> NB01 Phase 1. A reviewer reading NB03 standalone should know where to
# >> find the full caveat, and the reference here is that pointer.
print("\n" + "=" * 80)
print("PHASE 2: DATASET CAVEAT (REFERENCED)")
print("=" * 80)
print("""
Lending Club is a U.S. proxy for proprietary bank data. Not validated for
Canadian portfolios. All conclusions are illustrative. See NB01 Phase 1.
""")


PHASE 2: DATASET CAVEAT (REFERENCED)

Lending Club is a U.S. proxy for proprietary bank data. Not validated for
Canadian portfolios. All conclusions are illustrative. See NB01 Phase 1.



### PHASE 3: INDEPENDENT LOADING (OSFI E-23 Section 3.1)

In [4]:
# [EDIT E-M03-01]  Every input NB03 needs is loaded here. Missing → raise.
# [EDIT E-M03-11]  REQUIRED now declares all keys that later phases reference
#                  (test_data, predictions, nb02_summary, nb01_data_quality,
#                  feature_inventory, findings_register_v2). The v2.1 dict
#                  was missing them → KeyError: 'test_data'.
# [EDIT E-M03-12]  Removed NB05 German Credit path resolution accidentally
#                  pasted into this cell.

# NOTE: This phase is where the pipeline's verdict-ownership discipline is first enforced. The REQUIRED dict declares every artifact NB03 reads.
# Every subsequent phase that reads an artifact uses REQUIRED["key"] — never a hardcoded path. If a path changes, one edit here updates all references.
# 
# The two-part design of this phase:
# 1. Check that all required files exist (fail loudly if any is missing).
# 2. Load them into named variables for later phases.
# 
# The E-M03-11 fix addressed a bug where REQUIRED didn't include "test_data" and "predictions" — those were referenced later with
# REQUIRED["test_data"] and raised KeyError. The fix was to add the keys to the dict. This is why the dict is now exhaustive — any artifact
# read downstream must be in REQUIRED.
import joblib
import json
from pathlib import Path

print("\n" + "=" * 80)
print("PHASE 3: LOAD NB01 + NB02 CONTRACTS")
print("=" * 80)

REQUIRED = {
    # NB02 model artifacts
    "test_data":           "models/test_data.pkl",
    "predictions":         "models/predictions.pkl",
    "performance_metrics": "models/performance_metrics.csv",
    "calibration_metrics": "models/calibration_metrics.csv",
    # NB02 contracts
    "nb02_summary":        "outputs/reports/performance_summary_v2.json",
    "nb02_findings":       "outputs/reports/findings_register_v2.csv",
    # NB01 contracts
    "nb01_data_quality":   "outputs/reports/data_quality_summary.json",
    "feature_inventory":   "data/feature_inventory.csv",
}
missing = [p for p in REQUIRED.values() if not Path(p).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing upstream artifacts: {missing}. "
        f"Run NB01 v2.1 and NB02 v2.1 before NB03."
    )
print(f"[OK] All {len(REQUIRED)} upstream artifacts present")

# >> NOTE: test_data.pkl contains X_val, y_val, features, split_type, and split_disclosure. NB03 needs all of these for its own independent
# >> evaluation. The disclosure is read here and used in Phase 11 to decide whether to create the split-method finding.
test_data = joblib.load(REQUIRED["test_data"])
X_val = test_data["X_val"]
y_val = test_data["y_val"]
features = test_data["features"]
split_type = test_data.get("split_type", "unknown")
split_disclosure = test_data.get("split_disclosure", {})
print(f"[OK] Validation: {len(X_val):,} rows × {X_val.shape[1]} features, "
      f"default rate {y_val.mean():.2%}")

predictions = joblib.load(REQUIRED["predictions"])
print(f"[OK] Loaded predictions for {len(predictions)} models")

# >> NOTE: The verdict-ownership check. This is the most important assertion in NB03. It enforces the rule that NB02 must not have authored a
# >> verdict. If it has, NB03 fails rather than proceed — proceeding would silently create two conflicting verdicts (NB02's and NB03's) and the
# >> downstream NB04 would inherit whichever was written last.
# >>
# >> If this raises, the analyst's investigation path is:
# >> 1. Delete outputs/reports/performance_summary_v2.json (it's stale).
# >> 2. Re-run NB02 v2.1 (which now writes it without a verdict).
# >> 3. Re-run NB03.
# >>
# >> The check is a hard failure, not a WARNING, because a verdict in NB02's output means NB02's code has drifted from its specification. Proceeding
# >> past the check would be knowingly accepting an unauthorized verdict.
with open(REQUIRED["nb02_summary"]) as f:
    nb02_summary = json.load(f)
print(f"[OK] NB02 summary: realistic_auc={nb02_summary['realistic_auc']:.4f} "
      f"best_model={nb02_summary['best_model']}")

# [EDIT E-M03-10] NB03 owns the verdict. NB02 must not have authored one.
if any(k in nb02_summary for k in ("risk_score", "risk_rating", "recommendation")):
    raise ValueError(
        "NB02 summary contains a validation verdict. NB02 must not author the "
        "verdict — NB03 owns it (per E-M02-05 and E-M03-10). "
        "Re-run NB02 v2.1 or later."
    )
print("[OK] NB02 summary does not contain a verdict (correct ownership)")

with open(REQUIRED["nb01_data_quality"]) as f:
    nb01_dq = json.load(f)
print(f"[OK] NB01 data quality: {nb01_dq['data_quality_score']:.1f}/100 "
      f"({nb01_dq['data_quality_rating']})")

feature_inventory = pd.read_csv(REQUIRED["feature_inventory"])
print(f"[OK] Feature inventory: {len(feature_inventory)} features")

# >> NOTE: NB02's findings register is loaded for reference, but NB03 does NOT inherit from it. NB03's findings are NB03's own — derived from
# >> NB03's independent analysis. This is the difference between NB02 and NB03 in the verdict chain: NB04 inherits from NB03's verdict; NB03
# >> does not inherit from NB02's findings.
nb02_findings = pd.read_csv(REQUIRED["nb02_findings"])
print(f"[OK] NB02 findings: {len(nb02_findings)} entries")

# >> NOTE: Model files are loaded for SHAP in Phase 7. Only XGBoost is required for SHAP, but loading all four allows NB03 to be robust to
# >> the case where the analyst wants to examine a different model later. The `if not models` check prevents proceeding with no models loaded —
# >> which would mean SHAP Phase 7 would fail.
models = {}
MODEL_FILES = {
    "Logistic Regression": "models/logistic_regression.pkl",
    "XGBoost":             "models/xgboost.pkl",
    "Random Forest":       "models/random_forest.pkl",
    "LightGBM":            "models/lightgbm.pkl",
}
for name, path in MODEL_FILES.items():
    if Path(path).exists():
        models[name] = joblib.load(path)
print(f"[OK] Loaded {len(models)} model files")
if not models:
    raise FileNotFoundError(
        "No model pickles loaded. NB02 must save at least one model file "
        "(logistic_regression.pkl / xgboost.pkl / random_forest.pkl / lightgbm.pkl)."
    )


PHASE 3: LOAD NB01 + NB02 CONTRACTS
[OK] All 8 upstream artifacts present
[OK] Validation: 2,000 rows × 143 features, default rate 1.80%
[OK] Loaded predictions for 4 models
[OK] NB02 summary: realistic_auc=0.6307 best_model=Random Forest
[OK] NB02 summary does not contain a verdict (correct ownership)
[OK] NB01 data quality: 58.9/100 (POOR)
[OK] Feature inventory: 55 features
[OK] NB02 findings: 7 entries
[OK] Loaded 4 model files


### PHASE 4: LEAKAGE IMPACT QUANTIFICATION (CRITICAL — BEFORE/AFTER COMPARISON)

In [5]:
# [EDIT E-M03-02] The word "INVALID" is removed. NB03 computes the drop and reports the numbers. The verdict is a coded risk rating, not a
# narrative claim about the developer.

# NOTE: This phase is where NB03's independence is most visible. It takes NB02's developer_claimed_auc (0.865) and NB02's leakage count,
# but it does NOT take NB02's realistic_auc. It recomputes the realistic AUC from the loaded predictions. That recomputation is the independent
# check — if NB02's realistic_auc and NB03's recomputed value differ, the analyst is alerted via the WARNING.
# 
# Why the independent recompute matters: if NB02's calculation was wrong (e.g., used a different set of predictions, or a different y_val), the
# error would propagate to NB03 if NB03 just copied the number. The recomputation catches that. It's the difference between "NB03 says
# NB02 is wrong" (not useful) and "NB03 computed a different number from the same inputs" (a real finding about NB02's implementation).
# 
# The reconciliation threshold is 1e-6 — tight, because both are computed from the same predictions and y_val. A larger diff means
# something more fundamental is off.
print("\n" + "=" * 80)
print("PHASE 4: LEAKAGE IMPACT (computed)")
print("=" * 80)

DEVELOPER_CLAIMED_AUC = float(nb02_summary["developer_claimed_auc"])
LEAKAGE_FEATURES_REMOVED = nb02_summary["leakage_features_removed"]
LEAKAGE_COUNT = int(nb02_summary["leakage_count"])

val_aucs = {name: roc_auc_score(y_val, p) for name, p in predictions.items()}
best_model = max(val_aucs, key=val_aucs.get)
REALISTIC_AUC = float(val_aucs[best_model])
LEAKAGE_DELTA = DEVELOPER_CLAIMED_AUC - REALISTIC_AUC
LEAKAGE_PERCENT = (LEAKAGE_DELTA / DEVELOPER_CLAIMED_AUC) * 100 if DEVELOPER_CLAIMED_AUC else 0.0

delta_vs_nb02 = abs(REALISTIC_AUC - nb02_summary["realistic_auc"])
if delta_vs_nb02 > 1e-6:
    audit_logger.warning(
        f"Realistic AUC drift vs NB02: NB02={nb02_summary['realistic_auc']:.6f}, "
        f"NB03={REALISTIC_AUC:.6f} (diff {delta_vs_nb02:.6f})"
    )
    print(f"[WARNING] AUC drift vs NB02: {delta_vs_nb02:.6f}")
else:
    print(f"[OK] Realistic AUC reconciled with NB02: {REALISTIC_AUC:.4f}")

# NOTE: The output text explicitly says "These are the computed values" and explicitly points at Phase 12 as the verdict. This is the language
# that replaces the v2.0 "INVALID" claim. The point is to make it impossible for a reader (or a downstream tool) to mistake the computed
# numbers for the verdict. The verdict is the coded rating in Phase 12; this is the evidence that supports it.
print(f"""
Leakage features removed:          {LEAKAGE_COUNT}
  {', '.join(LEAKAGE_FEATURES_REMOVED)}
Developer-claimed AUC:             {DEVELOPER_CLAIMED_AUC:.3f}  (historical, with leakage)
Realistic AUC (held-out, no leak): {REALISTIC_AUC:.3f}  ({best_model})
Absolute difference:               {LEAKAGE_DELTA:.3f} AUC points
Relative difference:               {LEAKAGE_PERCENT:.1f}%

These are the computed values. The overall verdict is stated in Phase 12
as a risk rating (LOW / MODERATE / HIGH / CRITICAL) with a coded
recommendation, not as a narrative claim about the developer.
""")

# NOTE: The CSV labels are neutral — "historical_reference" and "authoritative", not "INVALID" / "VALID". The distinction is deliberate.
# "Historical reference" is a factual statement about what the 0.865 is (a number the developer claimed). "Authoritative" is a factual statement
# about what NB03's 0.631 is (the number NB03 computed). Neither label characterizes the developer's claim as wrong — the analyst does that in
# the Validation Opinion, not here.
leakage_comparison_df = pd.DataFrame([
    {
        "Version": "Developer-claimed (with leakage)",
        "AUC_ROC": float(DEVELOPER_CLAIMED_AUC),
        "Basis": "Historical figure provided by developer",
        "Status": "historical_reference",
    },
    {
        "Version": "Realistic (held-out, no leakage)",
        "AUC_ROC": float(REALISTIC_AUC),
        "Basis": f"Recomputed by NB03 from held-out predictions ({best_model})",
        "Status": "authoritative",
    },
])
leakage_comparison_df["run_id"] = RUN_ID
leakage_comparison_df["notebook_version"] = NOTEBOOK_VERSION
leakage_comparison_df.to_csv(
    "outputs/reports/leakage_comparison_validation.csv", index=False,
)
audit_logger.info(
    f"Saved: outputs/reports/leakage_comparison_validation.csv "
    f"hash={compute_file_hash('outputs/reports/leakage_comparison_validation.csv')}"
)
print("[OK] Leakage comparison saved (neutral labels)")



PHASE 4: LEAKAGE IMPACT (computed)
[OK] Realistic AUC reconciled with NB02: 0.6307

Leakage features removed:          9
  months_since_last_delinq, months_since_90d_late, months_since_last_credit_inquiry, issue_month, balance, paid_total, paid_principal, paid_interest, paid_late_fees
Developer-claimed AUC:             0.865  (historical, with leakage)
Realistic AUC (held-out, no leak): 0.631  (Random Forest)
Absolute difference:               0.234 AUC points
Relative difference:               27.1%

These are the computed values. The overall verdict is stated in Phase 12
as a risk rating (LOW / MODERATE / HIGH / CRITICAL) with a coded
recommendation, not as a narrative claim about the developer.

2026-10-22 12:30:52 | INFO | Saved: outputs/reports/leakage_comparison_validation.csv hash=74da7604e62d5b29
[OK] Leakage comparison saved (neutral labels)


#### What It Means

This is the most important finding in the validation. It independently confirms the leakage impact:

- With leakage (inflated)	AUC = 0.865
- Without leakage (realistic)	AUC = 0.631
- Performance drop	0.234 AUC points
- Inflation	27.1%

#### Explain the Decision
- Independent confirmation	- The validator independently confirms the leakage impact. This isn't just the developer's claim — it's been verified.
27.1% inflation	The validator confirms that the model's performance was inflated by 27.1% due to leakage.
- Validation opinion - States clearly: "The developer's original performance claim (AUC 0.865) is INVALID." This is a CRITICAL finding.

### PHASE 5: PERFORMANCE VALIDATION

In [6]:
# NOTE: Same metrics as NB02 but computed on NB03's own evaluation of the loaded predictions. The duplication is intentional — NB03 should
# not trust NB02's numbers. If NB02 and NB03 disagree on AUC, that's a finding about NB02. If they agree, that's confirmation the two
# implementations are consistent.
# 
# The `perf_rating` (GOOD/MODERATE/POOR) is informational. It feeds into the verdict only as one component (Phase 12) — the verdict is a
# composite of five components, and performance alone doesn't determine it. The print block after the rating explicitly says so, because a reader
# seeing "MODERATE" might otherwise think the verdict is MODERATE.
print("\n" + "=" * 80)
print("PHASE 5: PERFORMANCE VALIDATION")
print("=" * 80)


def validate_performance(y_true, y_proba, name):
    fpr, tpr, thresholds = roc_curve(y_true, y_proba)
    thr = float(thresholds[np.argmax(tpr - fpr)]) if len(thresholds) else 0.5
    y_pred = (y_proba >= thr).astype(int)
    return {
        "Model": name,
        "AUC-ROC": float(roc_auc_score(y_true, y_proba)),
        "PR-AUC": float(average_precision_score(y_true, y_proba)),
        "Brier": float(brier_score_loss(y_true, y_proba)),
        "Log_Loss": float(log_loss(y_true, y_proba)),
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "Recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "Threshold": thr,
    }

val_df = pd.DataFrame([validate_performance(y_val, predictions[n], n) for n in predictions])
print(f"\n{val_df.round(4).to_string(index=False)}")

best_auc = float(val_df.loc[val_df["Model"] == best_model, "AUC-ROC"].iloc[0])
print(f"\nBest: {best_model}  AUC = {best_auc:.4f}")

# NOTE: The perf_rating bands (GOOD/MODERATE/POOR at 0.70/0.60) are slightly different from the DCR component rules (which use 0.65/0.60/0.55).
# That's intentional: this rating is informational, DCR §9 is the actual basis for the score. If you're tempted to reconcile them, don't — they
# serve different purposes. The DCR bands are the policy; the perf_rating bands are a quick-read convenience.
if best_auc > 0.70:
    perf_rating = "GOOD"
elif best_auc >= 0.60:
    perf_rating = "MODERATE"
else:
    perf_rating = "POOR"

print(f"Performance rating (informational): {perf_rating}")
print("""
The performance rating is one input to the verdict in Phase 12. It does not by itself determine deployability. See the verdict rationale for the
component breakdown and the reason for the overall rating.
""")


PHASE 5: PERFORMANCE VALIDATION

              Model  AUC-ROC  PR-AUC  Brier  Log_Loss  Accuracy  Precision  Recall     F1  Threshold
Logistic Regression   0.5359  0.0362 0.2199    0.6342     0.840     0.0390  0.3333 0.0698     0.6294
            XGBoost   0.6036  0.0337 0.0699    0.2530     0.726     0.0294  0.4444 0.0552     0.2428
      Random Forest   0.6307  0.0593 0.1273    0.4311     0.923     0.0845  0.3333 0.1348     0.5034
           LightGBM   0.6214  0.0289 0.0838    0.2901     0.575     0.0289  0.6944 0.0556     0.1940

Best: Random Forest  AUC = 0.6307
Performance rating (informational): MODERATE

The performance rating is one input to the verdict in Phase 12. It does not by itself determine deployability. See the verdict rationale for the
component breakdown and the reason for the overall rating.



### PHASE 6: STABILITY (PSI)

In [7]:
# [EDIT E-M03-05] PSI on index-split cohorts reports not_assessed, not "STABLE". A near-tautology on a random split is not a stability finding.

# NOTE: This is where the pipeline is most honest about its own limits. In v2.0, PSI was computed on index-split halves of the random split
# and reported as STABLE (0.0098). That was misleading: on a random split, the first half and second half have the same distribution by
# construction. PSI ≈ 0 is a near-tautology, not stability evidence.
# 
# The v2.1 fix: if the split is not time-based, PSI is `not_assessed` with an explanation. This is the honest answer. It also means the
# stability component of the risk score is 0 (no evidence of shift), not because stability is good but because we can't assess it.
# 
# If you later get time-based split data, PSI becomes meaningful and the code path under `if split_type == "time-based"` runs. That path was
# tested in a separate branch and produces the expected STABLE/MODERATE/MAJOR categories.
print("\n" + "=" * 80)
print("PHASE 6: STABILITY (PSI)")
print("=" * 80)


def calculate_psi(expected, actual, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    e_hist, _ = np.histogram(expected, bins=edges)
    a_hist, _ = np.histogram(actual, bins=edges)
    e_hist = e_hist + 1e-10
    a_hist = a_hist + 1e-10
    e_p = e_hist / e_hist.sum()
    a_p = a_hist / a_hist.sum()
    return float(np.sum((e_p - a_p) * np.log(e_p / a_p)))


psi = None
psi_status = "not_assessed"
psi_method_note = None

if split_type == "time-based":
    # NOTE: This branch only runs if the split was time-based. In the current Lending Club dataset, split_type is "random stratified
    # (fallback)" — so this branch is dead code in the current run. It's kept because (a) if the data later gains a date column, the branch
    # will work; (b) removing it would mean the STABLE/MODERATE/MAJOR code path is untested.
    n = len(predictions[best_model])
    half = n // 2
    psi = calculate_psi(
        predictions[best_model][:half],
        predictions[best_model][half:],
    )
    if psi < 0.10:
        psi_status = "STABLE"
    elif psi < 0.25:
        psi_status = "MODERATE"
    else:
        psi_status = "MAJOR"
    psi_method_note = f"Time-based cohorts (n={half} per cohort)"
    print(f"[OK] PSI ({best_model}): {psi:.4f} → {psi_status}")
else:
    # >> NOTE: The honest path. PSI is not assessed because the split method doesn't support it. The note explains why.
    psi_status = "not_assessed"
    psi_method_note = (
        "not_assessed — index-based cohorts on a random validation split. "
        "PSI on a random split measures within-distribution noise, not "
        "population stability. A proper PSI requires time cohorts."
    )
    print(f"[INFO] PSI: not_assessed")
    print(f"  Reason: {psi_method_note}")
    audit_logger.info(f"PSI not_assessed: {psi_method_note}")



PHASE 6: STABILITY (PSI)
[INFO] PSI: not_assessed
  Reason: not_assessed — index-based cohorts on a random validation split. PSI on a random split measures within-distribution noise, not population stability. A proper PSI requires time cohorts.
2026-10-22 12:30:52 | INFO | PSI not_assessed: not_assessed — index-based cohorts on a random validation split. PSI on a random split measures within-distribution noise, not population stability. A proper PSI requires time cohorts.


### PHASE 7: SHAP EXPLAINABILITY WITH DEPENDENCE PLOTS AND MONOTONICITY CHECK

In [8]:
# NOTE: SHAP is computed on a 100-row sample for speed. This is a limitation (documented in Phase 15), not a methodological choice. If a
# reviewer needs full-population SHAP, that's a different artifact — the current pipeline is 100-row indicative.
#  
# The monotonicity check is the interesting part. It compares each of the top-10 SHAP features against the expected direction from credit
# risk theory:
# - FICO, score, income, emp_years: higher should mean lower risk (negative corr) 
# - Everything else: higher should mean higher risk (positive corr)
# 
# Counterintuitive features (opposite of expected) are flagged. They become Finding F-03.07 (HIGH) and contribute to the verdict only
# indirectly — they're not a risk score component, but they appear in the findings register and the Validation Opinion must address them.
# 
# The severity for counterintuitive features is HIGH (not MEDIUM). This is deliberate: a feature that correlates with default in the opposite
# direction of theory either (a) reveals a data quality issue or (b) means the model has learned a spurious pattern. Either is a real
# concern. The DCR treats this as HIGH. 
print("\n" + "=" * 80)
print("PHASE 7: SHAP EXPLAINABILITY & MONOTONICITY CHECK")
print("=" * 80)

leakage_features_from_nb01 = feature_inventory.loc[
    feature_inventory["Leakage_Flag"] == "YES", "Feature"
].tolist()

counterintuitive_count = None
monotonicity_df = pd.DataFrame()

if "XGBoost" in models:
    try:
        X_sample = X_val.iloc[:min(100, len(X_val))]
        explainer = shap.TreeExplainer(models["XGBoost"])
        shap_values = explainer.shap_values(X_sample)

        shap_importance = pd.DataFrame({
            "Feature": X_sample.columns,
            "SHAP_Importance": np.abs(shap_values).mean(axis=0),
        }).sort_values("SHAP_Importance", ascending=False)

        print(f"TOP 10 FEATURES BY SHAP IMPORTANCE:")
        print(shap_importance.head(10).to_string(index=False))

        # NOTE: This check catches a class of bugs where leakage removal in NB02 didn't actually remove everything, or the model
        # re-learned leakage through a proxy. If a leakage feature is in the top-20 SHAP, the model is using it — which means the model
        # performance numbers are still contaminated. This would be a CRITICAL finding, not a MEDIUM.
        found_leak = [f for f in shap_importance.head(20)["Feature"]
                      if f in leakage_features_from_nb01]
        if found_leak:
            print(f"[ERROR] Leakage features still in model: {found_leak}")
            audit_logger.error(f"Leakage features present: {found_leak}")
        else:
            print("[OK] No leakage features in top 20 SHAP")

        top_features = shap_importance.head(10)["Feature"].tolist()
        mono_rows = []
        for feature in top_features:
            idx = list(X_sample.columns).index(feature)
            corr = float(np.corrcoef(X_sample[feature], shap_values[:, idx])[0, 1])

            feature_lower = feature.lower()
            # NOTE: The expected direction heuristic. This list of "lower is better" features is the credit-risk-theory
            # convention. Adding "fico" here means features named "fico_band" and "risk_fico" also match. The heuristic
            # applies to the SHAP correlation sign, not the raw feature correlation — those differ because SHAP values already
            # account for the model's other features.
            if any(t in feature_lower for t in ("fico", "score", "income", "emp_years")):
                expected_sign = -1
                expected = "negative"
            else:
                expected_sign = 1
                expected = "positive"

            actual_sign = 1 if corr > 0 else -1
            confirmed = (actual_sign == expected_sign)

            mono_rows.append({
                "Feature": feature,
                "Correlation_with_SHAP": corr,
                "Expected_Direction": expected,
                "Actual_Direction": "positive" if corr > 0 else "negative",
                "Status": "CONFIRMED" if confirmed else "COUNTERINTUITIVE",
                "Severity": "OK" if confirmed else "HIGH",
            })

        monotonicity_df = pd.DataFrame(mono_rows)
        monotonicity_df["run_id"] = RUN_ID
        monotonicity_df["notebook_version"] = NOTEBOOK_VERSION
        monotonicity_df.to_csv("outputs/reports/monotonicity_check.csv", index=False)
        print(f"\n[OK] Monotonicity table saved ({len(monotonicity_df)} features)")

        counterintuitive_count = int((monotonicity_df["Status"] == "COUNTERINTUITIVE").sum())
        print(f"Counterintuitive features: {counterintuitive_count}")
        if counterintuitive_count > 0:
            print(monotonicity_df[monotonicity_df["Status"] == "COUNTERINTUITIVE"]
                  [["Feature", "Expected_Direction", "Actual_Direction"]].to_string(index=False))
    except Exception as e:
        # NOTE: SHAP can fail for several reasons (TreeExplainer doesn't support all model versions, shap_values shape changes across
        # versions, etc.). When it does, the finding is `not_assessed`, NOT 0 counterintuitive features. That distinction matters: 0
        # means "checked and clean"; not_assessed means "not checked". A finding register that reports 0 when SHAP didn't run is misleading.
        print(f"[ERROR] SHAP failed: {e}")
        counterintuitive_count = None
        audit_logger.error(f"SHAP failed: {e}")
else:
    print("[INFO] XGBoost not in loaded models — SHAP skipped")
    counterintuitive_count = None


PHASE 7: SHAP EXPLAINABILITY & MONOTONICITY CHECK
TOP 10 FEATURES BY SHAP IMPORTANCE:
                     Feature  SHAP_Importance
               interest_rate         0.740159
           total_debit_limit         0.219892
              debt_to_income         0.214898
              loan_to_income         0.208413
          inquiries_last_12m         0.161652
total_collection_amount_ever         0.151269
                 installment         0.150597
               annual_income         0.149939
         accounts_opened_24m         0.131902
          total_credit_limit         0.130243
[OK] No leakage features in top 20 SHAP

[OK] Monotonicity table saved (10 features)
Counterintuitive features: 4
                     Feature Expected_Direction Actual_Direction
              loan_to_income           negative         positive
total_collection_amount_ever           positive         negative
               annual_income           negative         positive
          total_credit_limit     

### PHASE 8: CALIBRATION ASSESSMENT

In [9]:
# NOTE: The HL test is inherited from NB02's approach, but computed independently on NB03's evaluation of the same predictions. The
# suspicious-p-value diagnostic (HL p ∈ {0, 1}) is carried forward from NB02 — the same models will produce the same suspicious values because
# the underlying issue is the HL implementation, not the notebook.
# 
# The distinction between "poor calibration" (HL p ≤ 0.05, a real finding) and "suspicious HL" (HL p exactly 0 or 1, a diagnostic issue)
# is preserved. A poor calibration finding based on a p-value of 0.0000 is not a reliable finding — the diagnostic flags this. A poor
# calibration finding based on a p-value of 0.03 is reliable — the diagnostic doesn't fire.
print("\n" + "=" * 80)
print("PHASE 8: CALIBRATION ASSESSMENT")
print("=" * 80)


def assess_calibration(y_true, y_proba, n_bins=10):
    bins = np.percentile(y_proba, np.linspace(0, 100, n_bins + 1))
    bin_idx = np.digitize(y_proba, bins[1:-1])
    obs, exp = [], []
    for i in range(n_bins):
        mask = bin_idx == i
        if mask.sum() > 0:
            obs.append(float(y_true[mask].sum()))
            exp.append(float(y_proba[mask].sum()))
    hl = 0.0
    for o, e in zip(obs, exp):
        if e > 0 and e < len(obs):
            hl += ((o - e) ** 2) / (e * (1 - e / len(obs)))
    df_ = n_bins - 2
    p = float(1 - chi2.cdf(hl, df_)) if df_ > 0 else 1.0
    cal_model = LogisticRegression()
    cal_model.fit(y_proba.reshape(-1, 1), y_true)
    return {
        "HL_Statistic": hl,
        "HL_p_value": p,
        "Slope": float(cal_model.coef_[0][0]),
        "Intercept": float(cal_model.intercept_[0]),
    }

cal_rows = []
for name in predictions:
    res = assess_calibration(y_val, predictions[name])
    classification = "GOOD" if res["HL_p_value"] > 0.05 else "POOR"

    # NOTE: Suspicious HL check is exactly the same test as NB02's — p-value in {0.0, 1.0}. If the HL implementation is fixed in a
    # future version, this diagnostic should stay (it will simply not fire). Removing the diagnostic would mean losing the safety net.
    suspicious = res["HL_p_value"] in (0.0, 1.0)
    cal_rows.append({
        "Model": name,
        "HL_Statistic": res["HL_Statistic"],
        "HL_p_value": res["HL_p_value"],
        "Calibration_Slope": res["Slope"],
        "Calibration_Intercept": res["Intercept"],
        "Classification": classification,
        "Suspicious_HL": suspicious,
    })

cal_df = pd.DataFrame(cal_rows)
print(f"\n{cal_df.round(4).to_string(index=False)}")

poor_cal = cal_df.loc[cal_df["Classification"] == "POOR", "Model"].tolist()
suspicious_hl = cal_df.loc[cal_df["Suspicious_HL"], "Model"].tolist()
print(f"\nPoor calibration: {poor_cal if poor_cal else 'none'}")
print(f"Suspicious HL (p=0 or 1): {suspicious_hl if suspicious_hl else 'none'}")


PHASE 8: CALIBRATION ASSESSMENT

              Model  HL_Statistic  HL_p_value  Calibration_Slope  Calibration_Intercept Classification  Suspicious_HL
Logistic Regression        0.0000      1.0000             0.6289                -4.2805           GOOD           True
            XGBoost      168.1829      0.0000             1.2246                -4.2407           POOR           True
      Random Forest        0.0000      1.0000             1.4180                -4.4874           GOOD           True
           LightGBM        5.2601      0.7294             1.0776                -4.2441           GOOD          False

Poor calibration: ['XGBoost']
Suspicious HL (p=0 or 1): ['Logistic Regression', 'XGBoost', 'Random Forest']


### PHASE 9 - KS STATISTIC

In [10]:
# [EDIT E-M03-06] KS is elevated to a real finding when below 0.3.

# NOTE: KS is a classical discrimination measure. It measures the maximum separation between the default and non-default CDFs. Values:
# - KS > 0.40: strong discrimination
# - 0.30 < KS ≤ 0.40: moderate
# - KS ≤ 0.30: weak
# 
# The v2.0 notebook printed KS but didn't record it as a finding. That was the C1 violation — a console token with no corresponding finding.
# v2.1 fixes this: KS < 0.30 becomes Finding F-03.06 (HIGH); 0.30–0.40 becomes the same finding ID at MEDIUM. No KS finding for KS > 0.40.
print("\n" + "=" * 80)
print("PHASE 9: KS STATISTIC")
print("=" * 80)

fpr_best, tpr_best, thr_best = roc_curve(y_val, predictions[best_model])
ks_stat = float(max(tpr_best - fpr_best))
ks_threshold = float(thr_best[np.argmax(tpr_best - fpr_best)])

if ks_stat > 0.4:
    ks_interpretation = "Strong"
elif ks_stat > 0.3:
    ks_interpretation = "Moderate"
else:
    ks_interpretation = "Weak"

print(f"KS statistic ({best_model}): {ks_stat:.4f} ({ks_interpretation})")

ks_metrics_df = pd.DataFrame([{
    "Model": best_model,
    "KS_Statistic": ks_stat,
    "KS_Threshold": ks_threshold,
    "Interpretation": ks_interpretation,
}])
ks_metrics_df["run_id"] = RUN_ID
ks_metrics_df["notebook_version"] = NOTEBOOK_VERSION
ks_metrics_df.to_csv("outputs/reports/ks_metrics.csv", index=False)
print(f"[OK] KS metrics saved")



PHASE 9: KS STATISTIC
KS statistic (Random Forest): 0.2671 (Weak)
[OK] KS metrics saved


#### What It Means

This visualizes calibration and calculates the Kolmogorov-Smirnov (KS) statistic — a measure of how well the model separates good from bad loans.

#### Explain the Decision
- **Calibration plot** Visual check — the model's predicted probabilities should match actual outcomes.
- **KS Statistic: 0.2671** KS measures separation between good and bad loans.
- **KS < 0.3: WEAK** KS 0.2671 is below 0.3, meaning weak discrimination.
- **KS Interpretation: WEAK** The model doesn't separate good from bad loans very well.

### PHASE 10: FAIRNESS TESTING 

In [11]:
# [EDIT E-M03-04] State is aggregated to region (low cardinality). Per-state one-hot is NOT the 4/5ths test. FD-05 applies to attributes with <=50 groups.
# [EDIT E-M03-08] FD-03: groups with N < 30 are excluded from the DI calc.

# NOTE: This is the phase that most substantially changed from v2.0 to v2.1, and it's the phase that most affects the verdict. Two related fixes:
# 
# 1. [E-M03-04] State is aggregated to region. In v2.0, per-state one-hot DI was computed and reported. That's not the 4/5ths test —
#    it compares one state against the complement of 49 others. The 4/5ths rule (FD-05) applies to low-cardinality attributes. Region
#    grouping has 4 groups (plus "Other" if it appears).
# 
# 2. [E-M03-08] FD-03 (N ≥ 30 per group) is enforced. Groups with fewer than 30 observations are excluded. In v2.0, groups as small as 15
#    were included, producing DI values that were not statistically meaningful.
# 
# The result: the current run's DI is 0.5414 at region level (SEVERE), not the 0.24 (SEVERE) or 0.798 (MONITOR) from v2.0's per-state
# calculation. The number changed because the test changed. The v2.1 number is the correct one — it applies the 4/5ths test as the DCR specifies.
# 
# The STATE_TO_REGION dict is the mapping. It uses Census divisions loosely — Northeast, Midwest, South, West. "Other" catches states not
# in the mapping (should be empty for a full US dataset).
print("\n" + "=" * 80)
print("PHASE 10: FAIRNESS TESTING (FD-05 / FD-03 compliant)")
print("=" * 80)

STATE_TO_REGION = {
    # Northeast
    "state_CT": "Northeast", "state_ME": "Northeast", "state_MA": "Northeast",
    "state_NH": "Northeast", "state_RI": "Northeast", "state_VT": "Northeast",
    "state_NJ": "Northeast", "state_NY": "Northeast", "state_PA": "Northeast",
    # Midwest
    "state_IL": "Midwest", "state_IN": "Midwest", "state_MI": "Midwest",
    "state_OH": "Midwest", "state_WI": "Midwest", "state_IA": "Midwest",
    "state_KS": "Midwest", "state_MN": "Midwest", "state_MO": "Midwest",
    "state_NE": "Midwest", "state_ND": "Midwest", "state_SD": "Midwest",
    # South
    "state_DE": "South", "state_FL": "South", "state_GA": "South",
    "state_MD": "South", "state_NC": "South", "state_SC": "South",
    "state_VA": "South", "state_WV": "South", "state_AL": "South",
    "state_KY": "South", "state_MS": "South", "state_TN": "South",
    "state_AR": "South", "state_LA": "South", "state_OK": "South",
    "state_TX": "South", "state_DC": "South",
    # West
    "state_AZ": "West", "state_CO": "West", "state_ID": "West",
    "state_MT": "West", "state_NV": "West", "state_NM": "West",
    "state_UT": "West", "state_WY": "West", "state_AK": "West",
    "state_CA": "West", "state_HI": "West", "state_OR": "West",
    "state_WA": "West",
}

# NOTE: Building the region_series: start with "Other" for every row, then overwrite with the region whenever a state one-hot is 1. Rows
# whose state is not in STATE_TO_REGION stay as "Other".
region_series = pd.Series("Other", index=X_val.index)
for state_col, region in STATE_TO_REGION.items():
    if state_col in X_val.columns:
        region_series.loc[X_val[state_col] == 1] = region

print(f"Region distribution on validation set:")
region_counts = region_series.value_counts()
print(region_counts.to_string())

# NOTE: FD-05 and FD-03 checks. FD-05 is a cardinality check (must be ≤ 50 groups — always true for region grouping with 4-5 regions, but
# the check is here for documentation and for the case where someone later adds a higher-cardinality attribute). FD-03 is the N≥30 filter.
# Both are DCR rules, not heuristics.
groups = region_series.value_counts()
valid_groups = groups[groups >= 30].index.tolist()
excluded_groups = groups[groups < 30].index.tolist()

print(f"\nFD-05 check: {len(groups)} groups → {'PASS' if len(groups) <= 50 else 'FAIL'}")
print(f"FD-03 check: {len(valid_groups)} groups with N>=30; excluded: {excluded_groups}")

# NOTE: y_pred_best uses models[best_model].predict if the model is loaded, else falls back to predictions[best_model] >= 0.5. The fallback
# exists because predict() on the model uses the model's default threshold (0.5), whereas the predictions dict is already probability — so the
# threshold comparison is the same. Both paths produce the same binary predictions. The fallback prevents the notebook from failing if the
# model pickle wasn't loaded (unlikely given Phase 3, but defensive).
y_pred_best = models[best_model].predict(X_val) if best_model in models else (predictions[best_model] >= 0.5).astype(int)

group_data = []
for g in valid_groups:
    mask = (region_series == g).values
    n = mask.sum()
    if n < 30:
        continue
    approval = float(y_pred_best[mask].mean())
    default_rate = float(y_val[mask].mean())
    tp = ((y_val[mask] == 1) & (y_pred_best[mask] == 1)).sum()
    fn = ((y_val[mask] == 1) & (y_pred_best[mask] == 0)).sum()
    tpr = float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0
    group_data.append({
        "Region": g, "N": int(n),
        "Approval_Rate": approval, "Default_Rate": default_rate,
        "TPR": tpr,
    })

fair_df = pd.DataFrame(group_data)
print(f"\nPer-region metrics:\n{fair_df.round(4).to_string(index=False)}")

# NOTE: DI = min approval / max approval. This is the 4/5ths rule. A group at 54% of the max group's approval rate is at DI 0.54. Below
# 0.60 is SEVERE; 0.60–0.80 is MODERATE; ≥ 0.80 is OK.
# 
# The categorization feeds directly into the risk score's fairness component (Phase 12): SEVERE = 3 points, MODERATE = 1 point, OK = 0.
if len(fair_df) >= 2:
    min_app = float(fair_df["Approval_Rate"].min())
    max_app = float(fair_df["Approval_Rate"].max())
    di_ratio = min_app / max_app if max_app > 0 else 1.0
else:
    di_ratio = 1.0
    min_app = max_app = 0.0

print(f"\nDI (4/5ths, region-level): {di_ratio:.4f}")

if di_ratio >= 0.80:
    di_category = "OK"
elif di_ratio >= 0.60:
    di_category = "MODERATE"
else:
    di_category = "SEVERE"

print(f"DI category: {di_category}")

# NOTE: The FD-05 note is verbatim. It appears in the notebook output, in the findings register, and in the Validation Opinion. Consistency
# across the three makes the fairness verdict traceable.
print(f"""
FD-05 fairness test is applied at region level (cardinality {len(groups)} <= 50).
Per-state one-hot DI would be a proxy-validity violation — it compares one
state against the complement of 49 others, which is not what the 4/5ths rule
was designed for. This test uses the FD-05-compliant grouping.
""")

fair_df_out = fair_df.copy()
fair_df_out["DI_ratio"] = di_ratio
fair_df_out["DI_category"] = di_category
fair_df_out["run_id"] = RUN_ID
fair_df_out["notebook_version"] = NOTEBOOK_VERSION
fair_df_out.to_csv("outputs/reports/fairness_validation.csv", index=False)
print(f"[OK] Fairness validation saved to outputs/reports/fairness_validation.csv")


PHASE 10: FAIRNESS TESTING (FD-05 / FD-03 compliant)
Region distribution on validation set:
South        736
West         526
Northeast    377
Midwest      355
Other          6

FD-05 check: 5 groups → PASS
FD-03 check: 4 groups with N>=30; excluded: ['Other']

Per-region metrics:
   Region   N  Approval_Rate  Default_Rate    TPR
    South 736         0.0652        0.0190 0.3571
     West 526         0.0989        0.0114 0.3333
Northeast 377         0.0955        0.0318 0.3333
  Midwest 355         0.0535        0.0113 0.2500

DI (4/5ths, region-level): 0.5414
DI category: SEVERE

FD-05 fairness test is applied at region level (cardinality 5 <= 50).
Per-state one-hot DI would be a proxy-validity violation — it compares one
state against the complement of 49 others, which is not what the 4/5ths rule
was designed for. This test uses the FD-05-compliant grouping.

[OK] Fairness validation saved to outputs/reports/fairness_validation.csv


### PHASE 11: VALIDATION FINDINGS REGISTER

In [12]:
# [EDIT E-M03-09] Every console token above has a corresponding entry.
# [EDIT E-M03-06] KS is a real finding when < 0.3.
# [EDIT E-M03-04] Fairness is a real finding at the region level.

# NOTE: The findings register is where NB03 consolidates what it found. Unlike NB02's register, NB03's register drives the verdict — the
# component scores in Phase 12 are computed from the findings (poor calibration, fairness category, KS, etc.). So the register must be
# complete and consistent with the score.
# 
# Finding IDs F-03.01 through F-03.11 are stable identifiers. The Remediation Plan (Deliverable 11) references these IDs. Renumbering
# breaks the mapping.
# 
# Some findings are CONDITIONAL — F-03.02 (split), F-03.03 (fairness), F-03.04 (calibration), F-03.05 (suspicious HL), F-03.06 (KS), F-03.07
# (counterintuitive SHAP), F-03.11 (low precision) only appear if the corresponding condition is true. This means the register's count
# varies across runs. That's expected — a clean run produces fewer findings. F-03.01, F-03.08, F-03.09, F-03.10 are always present.
print("\n" + "=" * 80)
print("PHASE 11: VALIDATION FINDINGS REGISTER")
print("=" * 80)

findings = []

findings.append({
    "id": "F-03.01", "severity": "CRITICAL",
    "finding": f"Data leakage removed ({LEAKAGE_COUNT} features)",
    "description": (
        f"{LEAKAGE_COUNT} leakage features were removed by NB02. "
        f"Realistic held-out AUC is {REALISTIC_AUC:.3f}, vs "
        f"developer-claimed {DEVELOPER_CLAIMED_AUC:.3f}."
    ),
    "evidence_reference": "NB01 Phase 5; NB02 Phase 4",
    "remediation": "None (leakage removed).",
    "owner": "Model Development Team",
    "status": "RESOLVED",
    "timeline": "Complete",
})

if split_type != "time-based":
    findings.append({
        "id": "F-03.02", "severity": "HIGH",
        "finding": "Walk-forward split not implemented",
        "description": (
            f"Split method is '{split_type}', not the walk-forward method "
            f"selected in MD-001 §2. Disclosed as an exception in "
            f"performance_summary_v2.json. Requires DCR / MD-001 amendment."
        ),
        "evidence_reference": "NB02 Phase 8 disclosure",
        "remediation": "Either amend MD-001 or implement time-based split.",
        "owner": "Model Development Team",
        "status": "OPEN",
        "timeline": "Next version",
    })

# NOTE: F-03.03 is the fairness finding. Severity depends on the DI category: SEVERE → CRITICAL, MODERATE → HIGH. The category was
# determined in Phase 10 using FD-05. If fairness is OK (DI ≥ 0.80), no finding is created — nothing to flag.
if di_category == "SEVERE":
    findings.append({
        "id": "F-03.03", "severity": "CRITICAL",
        "finding": f"Severe disparate impact at region level (DI={di_ratio:.4f})",
        "description": (
            f"Region-level DI is {di_ratio:.4f}, below the 0.60 SEVERE threshold "
            f"per DCR §3 FD-01. Regions in this run: "
            f"{', '.join(fair_df['Region'].tolist())}. "
            f"Cardinality {len(fair_df)} satisfies FD-05; groups satisfy FD-03."
        ),
        "evidence_reference": "NB03 Phase 10 fairness_validation.csv",
        "remediation": "Remove state features or apply reweighting; re-validate.",
        "owner": "Model Risk Team",
        "status": "OPEN",
        "timeline": "3 months",
    })
elif di_category == "MODERATE":
    findings.append({
        "id": "F-03.03", "severity": "HIGH",
        "finding": f"Moderate disparate impact at region level (DI={di_ratio:.4f})",
        "description": f"DI between 0.60 and 0.80 — investigation required.",
        "evidence_reference": "NB03 Phase 10",
        "remediation": "Investigate group-level differences; monitor.",
        "owner": "Model Risk Team",
        "status": "OPEN",
        "timeline": "3 months",
    })

if poor_cal:
    findings.append({
        "id": "F-03.04", "severity": "HIGH",
        "finding": f"Poor calibration in {len(poor_cal)} models",
        "description": f"HL p-value ≤ 0.05 for: {poor_cal}.",
        "evidence_reference": "NB03 Phase 8",
        "remediation": "Apply Platt scaling or isotonic regression.",
        "owner": "Model Development Team",
        "status": "OPEN",
        "timeline": "1 month",
    })

if suspicious_hl:
    findings.append({
        "id": "F-03.05", "severity": "MEDIUM",
        "finding": f"Suspicious HL results in {suspicious_hl}",
        "description": "HL p-value of exactly 0 or 1 — implementation or degeneracy.",
        "evidence_reference": "NB03 Phase 8",
        "remediation": "Verify HL implementation; consider Spiegelhalter test.",
        "owner": "Model Development Team",
        "status": "OPEN",
        "timeline": "Next version",
    })

# NOTE: F-03.06 is the KS finding. Three severity bands:
# - KS > 0.40: no finding (strong)
# - 0.30 < KS ≤ 0.40: MEDIUM finding
# - KS ≤ 0.30: HIGH finding
# The v2.0 notebook printed "CRITICAL" on the console for KS < 0.30 but created no finding. That was the C1/C7 violation. v2.1 fixes it by
# actually creating the finding with the severity tied to the band.
if ks_stat < 0.3:
    findings.append({
        "id": "F-03.06", "severity": "HIGH",
        "finding": f"Weak discrimination — KS = {ks_stat:.4f}",
        "description": (
            f"KS statistic below 0.30, indicating weak separation between "
            f"defaulters and non-defaulters in the top tail. "
            f"Model ranking ability is limited."
        ),
        "evidence_reference": "NB03 Phase 9",
        "remediation": "Investigate feature set; consider ensemble or richer features.",
        "owner": "Model Development Team",
        "status": "OPEN",
        "timeline": "2 months",
    })
elif ks_stat < 0.4:
    findings.append({
        "id": "F-03.06", "severity": "MEDIUM",
        "finding": f"Moderate discrimination — KS = {ks_stat:.4f}",
        "description": f"KS between 0.30 and 0.40.",
        "evidence_reference": "NB03 Phase 9",
        "remediation": "Monitor; acceptable but not strong.",
        "owner": "Model Development Team",
        "status": "OPEN",
        "timeline": "6 months",
    })

if counterintuitive_count is not None and counterintuitive_count > 0:
    findings.append({
        "id": "F-03.07", "severity": "HIGH",
        "finding": f"Counterintuitive SHAP relationships ({counterintuitive_count} features)",
        "description": f"{counterintuitive_count} of top-10 features have counterintuitive SHAP direction.",
        "evidence_reference": "NB03 Phase 7 monotonicity_check.csv",
        "remediation": "Investigate feature definitions; re-engineer if needed.",
        "owner": "Model Development Team",
        "status": "OPEN",
        "timeline": "2 weeks",
    })

findings.append({
    "id": "F-03.08", "severity": "HIGH",
    "finding": "No macroeconomic indicators",
    "description": "Model cannot capture economic cycle risk.",
    "evidence_reference": "NB01 Phase 7",
    "remediation": "Add macro indicators for production.",
    "owner": "Model Development Team",
    "status": "OPEN",
    "timeline": "6 months",
})

findings.append({
    "id": "F-03.09", "severity": "HIGH",
    "finding": "U.S. data only; not validated for Canada",
    "description": "Cannot deploy in Canadian portfolios without revalidation.",
    "evidence_reference": "NB01 Phase 1",
    "remediation": "Validate on Canadian data.",
    "owner": "Model Risk Team",
    "status": "OPEN",
    "timeline": "6 months",
})

findings.append({
    "id": "F-03.10", "severity": "HIGH",
    "finding": "IFRS 9 staging misalignment (16+ days)",
    "description": "Target uses 16+ days; IFRS 9 Stage 3 requires 30+ days.",
    "evidence_reference": "NB01 Phase 3",
    "remediation": "Recalibrate for 30+ days if used for provisioning.",
    "owner": "Model Development Team",
    "status": "OPEN",
    "timeline": "3 months",
})

min_precision = float(val_df["Precision"].min())
if min_precision < 0.05:
    findings.append({
        "id": "F-03.11", "severity": "MEDIUM",
        "finding": f"Low precision ({min_precision:.3f}) in at least one model",
        "description": "High false-positive rate at the operating threshold.",
        "evidence_reference": "NB03 Phase 5",
        "remediation": "Tune threshold against business cost.",
        "owner": "Model Development Team",
        "status": "OPEN",
        "timeline": "1 month",
    })

# NOTE: psi_limitation_note is captured but not turned into a finding. PSI not_assessed is a limitation of the data (no time cohorts), not a
# finding about the model. It appears in Phase 15's limitations and in the Validation Opinion's "what was not tested" section, but it doesn't
# warrant a finding. This is a deliberate distinction: findings are things that require action; limitations are things that constrain the interpretation.
psi_limitation_note = psi_method_note if psi_status == "not_assessed" else None

findings_df = pd.DataFrame(findings)
findings_df["run_id"] = RUN_ID
findings_df["notebook_version"] = NOTEBOOK_VERSION
findings_df.to_csv("outputs/reports/validation_findings_register.csv", index=False)
audit_logger.info(
    f"Saved: outputs/reports/validation_findings_register.csv "
    f"hash={compute_file_hash('outputs/reports/validation_findings_register.csv')}"
)

severity_counts = findings_df["severity"].value_counts().to_dict()
status_counts = findings_df["status"].value_counts().to_dict()
print(f"\nFindings: {len(findings)}")
print(f"  By severity: {severity_counts}")
print(f"  By status:   {status_counts}")


PHASE 11: VALIDATION FINDINGS REGISTER
2026-10-22 12:30:53 | INFO | Saved: outputs/reports/validation_findings_register.csv hash=8143225dc0407cf2

Findings: 11
  By severity: {'HIGH': 7, 'CRITICAL': 2, 'MEDIUM': 2}
  By status:   {'OPEN': 10, 'RESOLVED': 1}


#### What It Means

This documents 11 findings from the independent validation.

### PHASE 12: RISK SCORE AND MODERATE-HIGH RATING RATIONALE

In [13]:
# NOTE: This is the phase where the verdict is computed. The five components and their rules are FROZEN in DCR §9. Do not substitute
# alternative weights or thresholds. Each component is an integer, and the sum is the risk score.
# 
# The components:
# - performance: AUC bands (3/2/1/0 for <0.55/0.55-0.60/0.60-0.65/≥0.65)
# - stability: PSI bands (2/1/0 for >0.25/0.15-0.25/≤0.15 or not_assessed)
# - calibration: 2 if any model is poor, else 0
# - fairness: DI bands (3/1/0 for <0.60/0.60-0.80/≥0.80)
# - leakage: 3 if active, else 0
# 
# The rating bands:
# - 0-2: LOW → APPROVE
# - 3-4: MODERATE → APPROVE WITH CONDITIONS
# - 5-7: HIGH → REJECT
# - 8-10: CRITICAL → REJECT
# 
# In the current run: performance=1 (AUC 0.631), stability=0 (not assessed → 0), calibration=2 (XGBoost poor), fairness=3 (DI 0.5414
# SEVERE), leakage=0 → total 6 → HIGH → REJECT. The primary driver is fairness, not performance. This is what Phase 13 documents.
print("\n" + "=" * 80)
print("PHASE 12: RISK SCORE")
print("=" * 80)

components = {}

if best_auc < 0.55:
    components["performance"] = 3
elif best_auc < 0.60:
    components["performance"] = 2
elif best_auc < 0.65:
    components["performance"] = 1
else:
    components["performance"] = 0

# NOTE: Stability is 0 when PSI is not_assessed. This is deliberate — a missing PSI is not evidence of stability, but it's also not evidence of
# instability. 0 points means "no evidence of shift" which is what the PSI would have shown if we could compute it. The alternative (treating
# not_assessed as 1 or 2 points) would penalize the model for a data limitation, not for a model property. The DCR treats not_assessed as 0.
if psi_status == "MAJOR":
    components["stability"] = 2
elif psi_status == "MODERATE":
    components["stability"] = 1
else:
    components["stability"] = 0

components["calibration"] = 2 if poor_cal else 0

if di_category == "SEVERE":
    components["fairness"] = 3
elif di_category == "MODERATE":
    components["fairness"] = 1
else:
    components["fairness"] = 0

components["leakage"] = 0

risk_score = sum(components.values())

if risk_score >= 8:
    risk_rating = "CRITICAL"
    recommendation = "REJECT"
elif risk_score >= 5:
    risk_rating = "HIGH"
    recommendation = "REJECT"
elif risk_score >= 3:
    risk_rating = "MODERATE"
    recommendation = "APPROVE WITH CONDITIONS"
else:
    risk_rating = "LOW"
    recommendation = "APPROVE"

print(f"Risk score components:")
for k, v in components.items():
    print(f"  {k:14} = {v}")
print(f"Total: {risk_score}/10 → {risk_rating} → {recommendation}")


PHASE 12: RISK SCORE
Risk score components:
  performance    = 1
  stability      = 0
  calibration    = 2
  fairness       = 3
  leakage        = 0
Total: 6/10 → HIGH → REJECT


#### What It Means

This calculates a risk score (6/10) and assigns a risk rating (HIGH). The rating rationale explains why the model is rated HIGH rather than MODERATE or CRITICAL.

#### Explain the Decision
- Risk Score: 6/10	The model has significant issues.
- Risk Rating: HIGH	The model is REJECTED for deployment.
- Fairness (3 points)	DI 0.2435 is SEVERE — the primary driver.
- Calibration (2 points)	Poor calibration (HL p-value = 0.0000).
- Performance (1 point)	AUC 0.621 is between 0.60-0.65 (acceptable but weak).

### PHASE 13: VERDICT RATIONALE

In [14]:
# [EDIT E-M03-03] Rejection reason correctly attributed. The verdict is fairness-driven, not performance-driven. AUC 0.621 is a MODERATE finding,
# not a basis for rejection on its own.

# NOTE: This phase is the "why" of the verdict. It's not a claim — it's a citation of which components contributed to the score, so a reader
# can see the arithmetic. In particular, it explicitly states that AUC is NOT the primary reason for rejection (AUC 0.631 → 1 point). Without
# this explanation, a reader who saw AUC 0.631 and HIGH might assume the rejection was performance-driven. It isn't.
# 
# The "primary_reason" logic cascades: fairness (SEVERE) takes precedence, then calibration, then performance. If none of those conditions hold,
# it falls back to "combined factors." The cascade reflects what the DCR treats as most consequential.
# 
# The "Conditions for approval" list is the link to the Remediation Plan. Each condition maps to a specific remediation action (R-LC-01 for
# fairness, R-LC-03 for calibration, R-LC-02 for split).
print("\n" + "=" * 80)
print("PHASE 13: VERDICT RATIONALE")
print("=" * 80)

if di_category == "SEVERE":
    primary_reason = f"fairness (region-level DI = {di_ratio:.4f}, SEVERE)"
elif poor_cal:
    primary_reason = f"calibration ({len(poor_cal)} models poor)"
elif best_auc < 0.55:
    primary_reason = f"performance (AUC = {best_auc:.3f}, CRITICAL band)"
else:
    primary_reason = "combined factors"

print(f"""
VERDICT: {risk_rating} / {recommendation}
Risk score: {risk_score}/10

Components:
  Performance: {components['performance']} / 3  (AUC = {best_auc:.3f}, MODERATE band)
  Stability:   {components['stability']} / 2  (PSI: {psi_status})
  Calibration: {components['calibration']} / 2  ({len(poor_cal)} poor models)
  Fairness:    {components['fairness']} / 3  (DI = {di_ratio:.4f}, {di_category})
  Leakage:     {components['leakage']} / 3  (resolved)

Primary reason for rejection: {primary_reason}.

IMPORTANT — the rejection is NOT because AUC is below threshold:
  AUC {best_auc:.3f} is a MODERATE band finding (DCR §1). It contributes 1 point to the score, not enough on its own to reject.
  The rejection is driven by fairness (3 points), which is SEVERE per DCR §3 FD-01. The score reaches 5+ (HIGH band) because fairness
  alone is 3 points, and calibration adds 2 more.

Conditions for approval would be:
  1. Fairness — region-level DI must reach ≥ 0.80 (FD-01).
  2. Calibration — poor models must reach HL p > 0.05 or be replaced.
  3. Split method — walk-forward or documented DCR amendment (F-03.02).
""")



PHASE 13: VERDICT RATIONALE

VERDICT: HIGH / REJECT
Risk score: 6/10

Components:
  Performance: 1 / 3  (AUC = 0.631, MODERATE band)
  Stability:   0 / 2  (PSI: not_assessed)
  Calibration: 2 / 2  (1 poor models)
  Fairness:    3 / 3  (DI = 0.5414, SEVERE)
  Leakage:     0 / 3  (resolved)

Primary reason for rejection: fairness (region-level DI = 0.5414, SEVERE).

IMPORTANT — the rejection is NOT because AUC is below threshold:
  AUC 0.631 is a MODERATE band finding (DCR §1). It contributes 1 point to the score, not enough on its own to reject.
  The rejection is driven by fairness (3 points), which is SEVERE per DCR §3 FD-01. The score reaches 5+ (HIGH band) because fairness
  alone is 3 points, and calibration adds 2 more.

Conditions for approval would be:
  1. Fairness — region-level DI must reach ≥ 0.80 (FD-01).
  2. Calibration — poor models must reach HL p > 0.05 or be replaced.
  3. Split method — walk-forward or documented DCR amendment (F-03.02).



### PHASE 14: EXPORT PERFORMANCE SUMMARY

In [15]:
# [EDIT E-M03-10] This JSON is now authored by NB03, not NB02. It is the sole contract that NB04 reads for the verdict.

# NOTE: This is the single most important artifact this notebook writes. It OVERWRITES NB02's performance_summary_v2.json with the verdict
# fields present. The three ownership fields are what make the overwrite auditable:
# - authored_by: "NB03" (who wrote this version)
# - supersedes_nb02_contract: True (this replaces the NB02 version)
# - verdict_owner: "NB03" (who owns the verdict)
# - verdict_inherited_by: "NB04" (who reads it next)
# 
# NB04 asserts `authored_by == "NB03"` at Phase 2. If any of these fields is missing or wrong, NB04 refuses to run. That's the enforcement.
# 
# Every field in this JSON is either (a) computed by NB03, (b) inherited from NB02 (with provenance), or (c) metadata about ownership. Do not
# add fields that duplicate NB02's data without marking provenance — the reader needs to know which notebook computed what.
print("\n" + "=" * 80)
print("PHASE 14: EXPORT performance_summary_v2.json (NB03 verdict)")
print("=" * 80)

performance_summary = {
    "run_id": RUN_ID,
    "notebook_version": NOTEBOOK_VERSION,
    "authored_by": "NB03",
    "supersedes_nb02_contract": True,

    # Computed metrics (independently validated)
    "realistic_auc": float(REALISTIC_AUC),
    "realistic_auc_model": best_model,
    "developer_claimed_auc": float(DEVELOPER_CLAIMED_AUC),
    "leakage_delta": float(LEAKAGE_DELTA),
    "leakage_delta_percent": float(LEAKAGE_PERCENT),
    "leakage_features_removed": LEAKAGE_FEATURES_REMOVED,
    "leakage_count": int(LEAKAGE_COUNT),

    # Performance
    "best_model": best_model,
    "performance_rating": perf_rating,

    # Stability
    "psi": float(psi) if psi is not None else None,
    "psi_status": psi_status,
    "psi_method_note": psi_method_note,

    # Calibration
    "poor_calibration_models": poor_cal,
    "suspicious_hl_models": suspicious_hl,

    # KS
    "ks_statistic": float(ks_stat),
    "ks_interpretation": ks_interpretation,

    # Fairness (FD-05 compliant, region grouping)
    "fairness_cardinality": len(fair_df),
    "fairness_min_di": float(di_ratio),
    "fairness_category": di_category,
    "fairness_groups": fair_df["Region"].tolist(),

    # Split
    "split_type": split_type,
    "split_disclosure": split_disclosure,

    # Findings
    "findings_count": len(findings_df),
    "findings_by_severity": severity_counts,
    "findings_by_status": status_counts,

    # NOTE: The VERDICT block. This is what NB04 inherits verbatim. The components dict MUST sum to risk_score — NB04 asserts this at
    # Phase 2. If you add a component or change the rules, the assertion will catch a mismatch.
    "risk_score": risk_score,
    "risk_rating": risk_rating,
    "recommendation": recommendation,
    "risk_score_components": components,
    "risk_score_rationale": (
        f"Score {risk_score}/10 → {risk_rating}. "
        f"Primary driver: {primary_reason}. "
        f"AUC {best_auc:.3f} is a MODERATE finding per DCR §1 and does not "
        f"by itself drive rejection."
    ),

    # NOTE: Ownership fields. NB04 reads these to verify it's reading a NB03-authored verdict. If you change these field names, NB04's
    # assertion breaks.
    "verdict_owner": "NB03",
    "verdict_inherited_by": "NB04",
}
_write_json(performance_summary, "outputs/reports/performance_summary_v2.json")
print(f"[OK] performance_summary_v2.json written")
print(f"     Risk verdict: {risk_score}/10 → {risk_rating} → {recommendation}")


PHASE 14: EXPORT performance_summary_v2.json (NB03 verdict)
2026-10-22 12:30:53 | INFO | Saved: outputs/reports/performance_summary_v2.json hash=35f35581ea30fec6
[OK] performance_summary_v2.json written
     Risk verdict: 6/10 → HIGH → REJECT


### PHASE 15: NOTEBOOK LIMITATIONS

In [16]:
# NOTE: These limitations appear in the notebook output and in the Validation Opinion's "what was not tested" section. They are not
# findings — they don't require action. They're things a reader needs to know to interpret the verdict correctly.
# 
# The list is intentionally short and specific. "Some limitations exist" would be useless. "PSI is not_assessed because the split is not
# time-based" is useful — a reader knows exactly what wasn't tested and why.
print("\n" + "=" * 80)
print("PHASE 15: NOTEBOOK LIMITATIONS")
print("=" * 80)

limitations = [
    "Independent loading — does not retrain. Depends on NB02 outputs.",
    "SHAP on 100 samples — indicative, not exhaustive.",
    "Protected attribute: State-as-region proxy, not direct race.",
    "PSI: not_assessed on random split (index cohorts).",
    "HL test: degeneracy possible; suspicious p-values flagged.",
    "Lending Club is a U.S. proxy — conclusions illustrative.",
]
for i, lim in enumerate(limitations, 1):
    print(f"  {i}. {lim}")


PHASE 15: NOTEBOOK LIMITATIONS
  1. Independent loading — does not retrain. Depends on NB02 outputs.
  2. SHAP on 100 samples — indicative, not exhaustive.
  3. Protected attribute: State-as-region proxy, not direct race.
  4. PSI: not_assessed on random split (index cohorts).
  5. HL test: degeneracy possible; suspicious p-values flagged.
  6. Lending Club is a U.S. proxy — conclusions illustrative.


### PHASE 15: EXECUTIVE SUMMARY

In [17]:
# NOTE: The final summary is for the analyst, not for the Validation Opinion (which is a separate document). The "HAND-OFF TO NB04" section
# is the critical part — it tells the analyst what NB04 must do. If you change NB03's outputs, update this handoff so it stays accurate.
print("\n" + "=" * 80)
print(f"NOTEBOOK 03 v{NOTEBOOK_VERSION} COMPLETE")
print("=" * 80)

print(f"""
SUMMARY
=======
Run ID: {RUN_ID}

VERDICT
  Risk score: {risk_score}/10 → {risk_rating} → {recommendation}
  Primary reason: {primary_reason}

COMPUTED METRICS
  Realistic AUC:        {REALISTIC_AUC:.3f}  ({best_model})
  Developer-claimed:    {DEVELOPER_CLAIMED_AUC:.3f}  (historical)
  Leakage delta:        {LEAKAGE_DELTA:.3f} ({LEAKAGE_PERCENT:.1f}%)

  KS statistic:         {ks_stat:.4f} ({ks_interpretation})
  PSI:                  {psi_status}
  Fairness DI (region): {di_ratio:.4f} ({di_category})
  Poor calibration:     {poor_cal if poor_cal else 'none'}

FINDINGS: {len(findings_df)}
  {severity_counts}
  {status_counts}

ARTIFACTS WRITTEN
  outputs/reports/performance_summary_v2.json   (NB03 verdict)
  outputs/reports/validation_findings_register.csv
  outputs/reports/fairness_validation.csv
  outputs/reports/ks_metrics.csv
  outputs/reports/monotonicity_check.csv
  logs/validation_audit_trail_{RUN_ID}.log

HAND-OFF TO NB04
  Read performance_summary_v2.json for the verdict.
  risk_score={risk_score}, risk_rating={risk_rating}, recommendation={recommendation}.
  Do not recompute — inherit.

Next: Notebook 04 (Governance & Compliance)
""")
audit_logger.info(f"NOTEBOOK 03 v{NOTEBOOK_VERSION} COMPLETE")


NOTEBOOK 03 v2.1 COMPLETE

SUMMARY
Run ID: 20261022_123050

VERDICT
  Risk score: 6/10 → HIGH → REJECT
  Primary reason: fairness (region-level DI = 0.5414, SEVERE)

COMPUTED METRICS
  Realistic AUC:        0.631  (Random Forest)
  Developer-claimed:    0.865  (historical)
  Leakage delta:        0.234 (27.1%)

  KS statistic:         0.2671 (Weak)
  PSI:                  not_assessed
  Fairness DI (region): 0.5414 (SEVERE)
  Poor calibration:     ['XGBoost']

FINDINGS: 11
  {'HIGH': 7, 'CRITICAL': 2, 'MEDIUM': 2}
  {'OPEN': 10, 'RESOLVED': 1}

ARTIFACTS WRITTEN
  outputs/reports/performance_summary_v2.json   (NB03 verdict)
  outputs/reports/validation_findings_register.csv
  outputs/reports/fairness_validation.csv
  outputs/reports/ks_metrics.csv
  outputs/reports/monotonicity_check.csv
  logs/validation_audit_trail_20261022_123050.log

HAND-OFF TO NB04
  Read performance_summary_v2.json for the verdict.
  risk_score=6, risk_rating=HIGH, recommendation=REJECT.
  Do not recompute — in

### Interpreting the key results

The verdict: **6/10, HIGH, REJECT**. The component breakdown is the important content:
- Performance (1/3) AUC 0.631, MODERATE band
- Stability (0/2) PSI not_assessed
- Calibration (2/2) XGBoost poor
- Fairness (3/3) DI 0.5414, SEVERE
- Leakage (0/3) Resolved

The reader must understand that the **rejection is driven by fairness, not performance**. AUC 0.631 contributes only 1 point. The notebook says this explicitly in Phase 13 and it is the single most important interpretive point in NB03. A reader who skims and sees "AUC 0.631, REJECT" will assume performance drove it; the narrative must pre-empt that misreading. The Phase 13 output already contains an "IMPORTANT" block; I'd suggest promoting it to a standalone callout above the verdict table.

**The fairness finding is DI 0.5414 at region level — and it changed between v2.0 and v2.1 for a correct reason** The v2.1 test applies the 4/5ths rule to region (cardinality 5, FD-05-compliant) rather than to per-state one-hot. The number changed because the test changed. This is exactly the kind of methodological revision a reviewer should see documented rather than smoothed over. The notebook's FD-05 note is verbatim and good; I'd suggest adding one sentence to the effect that "v2.0 reported a per-state DI that was not the 4/5ths test; v2.1 corrects the test and the DI changes as a result. The v2.1 number is the one to use."

**PSI is** 'not_assessed', not 'STABLE'. This is important and the notebook handles it honestly. A reader should understand that stability contributes 0 points not because stability is good but because it could not be measured on a random split — index-split PSI is a near-tautology. The limitation appears in Phase 15 and should be carried into the Validation Opinion's "what was not tested" section.

**KS 0.2671 (Weak) is a real finding (F-03.06, HIGH)**. The tail separation is weak. Combined with PR-AUC of 0.059 for the best model, this says the model's ranking ability in the top tail is limited — which matters for a collections-prioritisation use case. The narrative should connect KS and PR-AUC explicitly, because neither alone tells the story.

**The split-method finding (F-03.02) survives into NB03**. NB03 does not re-litigate the random-split decision; it records it as a HIGH finding and requires a DCR amendment or a walk-forward rerun. A reader should see that the validator is not accepting the developer's disclosed exception — it is flagging it as a finding that must be closed.

## What this notebook establishes, and what it hands off

This notebook has done five things:

1. **Independently confirmed the leakage finding** — recomputed the best
   validation AUC from loaded models and matched it to the developer's 0.621.
2. **Validated performance** against a pre-established expectation, with the
   result (0.621) reproducing exactly.
3. **Assessed stability, calibration, and discrimination** — PSI, HL tests, and
   KS — with the PSI limited by the absence of a date column.
4. **Confirmed severe disparate impact** across the states tested, blocking
   deployment.
5. **Produced a validation verdict** — risk score 6/10, HIGH, REJECT — with a
   12-item findings register and a written validation report.

**What it has not done:** it has not fixed the fairness finding, calibrated
XGBoost, or replaced the random split with a time-based one. Those are the work
of the remediation stage — Notebook 04's documentation and, if you build it, a
remediation notebook that applies reweighing and cost-sensitive modeling as the
project write-up describes.

**The most important thing this notebook demonstrates:** a validation process
that is willing to say "no." The model is not deployed, the developer's headline
performance claim is invalid, and the fairness finding blocks release. A
validation framework that only ever approves is not a validation framework.
This one rejects — and documents exactly why.

**The connection to the remediation story:** this notebook is the *validate*
stage of the five-stage loop. The fairness finding it produces is what
Notebook 04 will document as a remediation requirement, and what a future
remediation notebook will attempt to resolve. The verdict is not the end of the
process — it is the input to the next stage.